# 11 · Production = M1.5: the checks

## Executive summary (read this first)

`forecast_models.py` (production, what the container runs) is now **M1.5**, the model of notebook 09. This notebook checks that
the two are the same model, number for number.

**Production reads only the unit folder.** The container ships three `.py` files and gets the unit's `card.toml` and panels; it
cannot read `transformations.toml`. So each asset's type comes from **the panel that holds it**: the parquet's `panel_id`
column, which is also its file name and the card's `[panels]` id.

| panel (unit folder) | type | step | shocks | drift |
|---|---|---|---|---|
| `rates_daily` | rates | change | Student-t | no |
| `g10_fx_daily`, `em_transfer_early` | fx | log change | Student-t | no |
| `factors_daily` (log_return card) | equity factor | the row itself | Student-t | no |
| `macro_monthly`: CPI_\*, PCE_\* | price index | log change | normal | **yes** |
| `macro_monthly`: NFP | payrolls | change | normal | **yes** |
| `macro_monthly`: UNRATE | unemployment | change | normal | no |
| anything else | M0's rule | change | normal | no |

A log_return card always steps by the row itself, whatever the panel.

**The checks:**
1. **Rules.** For every asset of the 103 units, production's rule (from the panel) equals notebook 09's (from
   `transformations.toml`).
2. **Draws at the as-of.** With the text neutral and seed = crc32(unit id), `forecast_models.build_draws` returns notebook 09's
   500 × assets × horizons draws on all 103 units. The largest difference is printed; 0 means identical.
3. **Draws on the backtest.** The same on every backtest date of the 99 daily units. On the 4 monthly units the notebooks use
   M0's fixed month table (built for the real as-of) at every past date, while production counts the months from the dates;
   the two agree at the as-of (check 2), so those units are checked there only.

So production's backtest score is notebook 09's: **0.9444 × M0** (the previous production walk, text off: 0.9726).

§4 shows what the text's adjustments do to production's draws. No realized outcome is read.

## 0 — Setup (as notebook 07), plus production

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

import zlib
import forecast_agent as fa, forecast_models as fm
NEUTRAL = {"shift": 0.0, "widen": 1.0, "skew": 0.0}
PANELS = {u: fa._read_panels(UNITS / u) for u in UNIT_IDS}
SEED = {u: zlib.crc32(UNITS_DATA[u]["id"].encode()) & 0x7FFFFFFF for u in UNIT_IDS}
MONTHLY = [u for u in UNIT_IDS if UNITS_DATA[u]["id"] in m0._MONTHLY_STEPS]
def production(u, origin, adj=None):
    U = UNITS_DATA[u]
    return fm.build_draws(PANELS[u], U["assets"], U["horizons"], origin, adj or {a: dict(NEUTRAL) for a in U["assets"]},
                          500, SEED[u], target_type=U["target_type"], family=U["family"])
print("monthly units:", MONTHLY)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}
monthly units: ['t2-F1-cpi-glidepath-2023', 't2-F1-sahm-watch-2024', 't2-F4-covid-nfp-2020', 't2-F4-cpi-vintage-2022']


## 1 — Notebook 09's model (M1.5), unchanged

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True, ewma_halflife=None):
    """ewma_halflife=None: the plain sd of the window (M1.3); a number: the EWMA sd with that halflife (M1.5)."""
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    if ewma_halflife is not None:                       # M1.5: EWMA sd per asset; correlation from the window
        sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
        sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])
        sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out


## 2 — Check 1: the rule per asset, production (panel in the unit folder) vs notebook 09 (`transformations.toml`)

In [3]:
rows = []
for u in UNIT_IDS:
    U = UNITS_DATA[u]
    for a in U["assets"]:
        panel_id, _ = fm._history(PANELS[u], a, U["asof"])
        prod = fm.asset_rule(a, panel_id, U["target_type"])
        rows.append({"panel id": panel_id, "type": fm.asset_type(a, panel_id), "target": U["target_type"], **prod,
                     "asset": a, "same as notebook 09": prod == U["rules"][a]})
RULES_CHECK = pd.DataFrame(rows)
print(f"{len(RULES_CHECK)} (unit, asset) pairs; production rule = notebook 09 rule on {RULES_CHECK['same as notebook 09'].sum()}")
assert RULES_CHECK["same as notebook 09"].all()
(RULES_CHECK.groupby(["panel id", "type", "target", "transform", "shock", "drift"])
 .agg(pairs=("asset", "size"), assets=("asset", lambda s: ", ".join(sorted(set(s))))))

165 (unit, asset) pairs; production rule = notebook 09 rule on 165


pairs                                            assets
panel id          type          target     transform shock     drift                                                         
em_transfer_early fx            level      log_diff  student_t False      6                                     BRL, CNY, INR
factors_daily     equity_factor log_return as_return student_t False     23                           HML, MKT, MOM, QMJ, SMB
g10_fx_daily      fx            level      log_diff  student_t False     60  AUD, CAD, CHF, DKK, EUR, GBP, JPY, NOK, NZD, SEK
macro_monthly     cpi           level      log_diff  normal    True       2                                           CPI_ALL
                  payrolls      level      diff      normal    True       1                                               NFP
                  unemployment  level      diff      normal    False      1                                            UNRATE
rates_daily       rates         level      diff      student_t False     72          UST_10Y, UST_2Y, UST_30Y, UST_5Y, UST_7Y

## 3 — Checks 2 and 3: the draws

In [4]:
asof_diff = pd.Series({u: float(np.max(np.abs(production(u, UNITS_DATA[u]["asof"]) - m13_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"], ewma_halflife=63))))
                      for u in UNIT_IDS})
print(f"check 2 — at the as-of, 103 units: largest |production - notebook 09| = {asof_diff.max():.1e} "
      f"(units with any difference: {(asof_diff > 0).sum()})")
assert asof_diff.max() == 0.0

def backtest_diff(u):
    U = UNITS_DATA[u]
    return u, len(BACKTEST[u]), max(float(np.max(np.abs(production(u, o) - m13_draws(U, o, ewma_halflife=63)))) for o, _ in BACKTEST[u])
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    BT = pd.DataFrame(pool.map(backtest_diff, [u for u in UNIT_IDS if u not in MONTHLY], chunksize=1),
                      columns=["unit", "dates", "largest difference"]).set_index("unit")
print(f"check 3 — every backtest date of the {len(BT)} daily units ({BT.dates.sum():,} dates, {time.time() - t0:.0f}s): "
      f"largest |production - notebook 09| = {BT['largest difference'].max():.1e}")
assert BT["largest difference"].max() == 0.0

check 2 — at the as-of, 103 units: largest |production - notebook 09| = 0.0e+00 (units with any difference: 0)


check 3 — every backtest date of the 99 daily units (23,593 dates, 157s): largest |production - notebook 09| = 0.0e+00


## 4 — What the text's adjustments do to production's draws

One F3 card (3 assets × 2 horizons), at its as-of. `shift` is added at every horizon in the asset's own units (also on FX,
which steps in logs); `widen` multiplies every deviation from the centre; `skew` tilts each leg's shock and keeps the mean and
sd in expectation (500 draws are noisy). The neutral row is notebook 09's M1.5.

In [5]:
u = "t2-F3-divergence-2014"; U = UNITS_DATA[u]; a = "EUR"; ia = U["assets"].index(a)
cases = {"neutral (= M1.5)": {}, "shift +0.02": {"shift": 0.02}, "widen 1.5": {"widen": 1.5}, "skew +0.5": {"skew": 0.5}}
TEXT = {}
for name, change in cases.items():
    adj = {x: dict(NEUTRAL) for x in U["assets"]} | {a: dict(NEUTRAL) | change}
    d = production(u, U["asof"], adj)[:, ia, :]
    for hi, h in enumerate(U["horizons"]):
        col = d[:, hi]; z = col - col.mean()
        TEXT[(name, h)] = {"mean": col.mean(), "sd": col.std(), "5%": np.quantile(col, 0.05), "95%": np.quantile(col, 0.95),
                           "skewness": float((z ** 3).mean() / z.std() ** 3)}
print(f"{u}, {a} (last value {U['rows'][a][-1][1]:.4f}), 500 draws:")
pd.DataFrame(TEXT).T.rename_axis(["text adjustment on EUR", "horizon"])

t2-F3-divergence-2014, EUR (last value 1.2628), 500 draws:


mean      sd    5%   95%  skewness
text adjustment on EUR horizon                                    
neutral (= M1.5)       63      1.263 0.03111 1.213  1.31    0.1578
                       126     1.266 0.04412 1.195 1.343    0.3721
shift +0.02            63      1.283 0.03111 1.233  1.33    0.1578
                       126     1.286 0.04412 1.215 1.363    0.3721
widen 1.5              63      1.264 0.04672 1.188 1.334     0.229
                       126     1.268 0.06648 1.163 1.385     0.448
skew +0.5              63      1.263 0.03364 1.215 1.329    0.7613
                       126     1.267 0.04781 1.197  1.35    0.8721